In [1]:
!pip show torch
!which python

Name: torch
Version: 2.10.0+cu128
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.12/dist-packages
Requires: cuda-bindings, filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvshmem-cu12, nvidia-nvtx-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, easyocr, fastai, kornia, peft, pytorch-ignite, pytorch-lightning, sentence-transformers, timm, torchaudio, torchdata, torchmetrics, torchvision
/usr/local/bin/python


In [2]:
!pip install transformers datasets sentencepiece accelerate sentence-transformers scikit-learn huggingface_hub

In [3]:
"""
train.py — NL-to-SQL Fine-tuning Script (v2.0)
================================================
Run this ONCE on Google Colab with a T4 GPU.

Improvements over v1.0:
  [1] Dynamic similarity thresholding   — replaces hardcoded top-3
  [2] Column-level schema linking        — embed/retrieve per column
  [3] Enriched schema representations   — PK/FK tags in BGE strings
  [4] Expanded context window           — max_input_length = 1024
"""


'\ntrain.py — NL-to-SQL Fine-tuning Script (v2.0)\n================================================\nRun this ONCE on Google Colab with a T4 GPU.\n\nImprovements over v1.0:\n  [1] Dynamic similarity thresholding   — replaces hardcoded top-3\n  [2] Column-level schema linking        — embed/retrieve per column\n  [3] Enriched schema representations   — PK/FK tags in BGE strings\n  [4] Expanded context window           — max_input_length = 1024\n'

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jeromeblanchet/yale-universitys-spider-10-nlp-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset


In [5]:
from huggingface_hub import notebook_login

notebook_login()

In [6]:
!pip show torch
!which python

Name: torch
Version: 2.10.0+cu128
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.12/dist-packages
Requires: cuda-bindings, filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvshmem-cu12, nvidia-nvtx-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, easyocr, fastai, kornia, peft, pytorch-ignite, pytorch-lightning, sentence-transformers, timm, torchaudio, torchdata, torchmetrics, torchvision
/usr/local/bin/python


In [7]:




import json
import os
import numpy as np
import torch
from dataclasses import dataclass
from typing import Optional

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from torch.utils.data import Dataset


In [10]:
@dataclass
class Config:
    # Paths
    train_path:     str = f"{path}/spider/train_spider.json"
    dev_path:       str = f"{path}/spider/dev.json"
    tables_path:    str = f"{path}/spider/tables.json"
    output_dir:     str = "content/t5_model"
    # Model
    model_name:     str = "google/flan-t5-base"
    bge_model_name: str = "BAAI/bge-base-en-v1.5"
    max_input_length:  int = 1024
    max_output_length: int = 256
    # Retrieval
    similarity_threshold: float = 0.45
    max_tables_fallback:  int   = 5
    top_k_columns: int = 12
    # Training
    epochs:                    int   = 10
    batch_size:                int   = 2    # ← was 8, reduce to 2
    gradient_accumulation_steps: int = 4   # ← add this, effective batch = 2×4 = 8
    learning_rate:             float = 5e-4
    warmup_steps:              int   = 500
    weight_decay:              float = 0.01
    num_beams:                 int   = 2    # ← was 4, halves beam search memory
    fp16:                      bool  = True


CFG = Config()

In [11]:

# ──────────────────────────────────────────────────────────────────────────────
# SCHEMA LOADER
# ──────────────────────────────────────────────────────────────────────────────

def load_schemas(tables_path: str) -> dict:
    """
    Returns a dict keyed by db_id.
    Each value is a dict with:
        tables   : list of table names
        columns  : dict  table_name → list of (col_name, col_type)
        pks      : dict  table_name → set of pk col names
        fks      : dict  table_name → dict  col_name → "other_table.other_col"
    """
    with open(tables_path) as f:
        raw = json.load(f)

    schemas = {}
    for db in raw:
        db_id   = db["db_id"]
        t_names = db["table_names_original"]  # e.g. ["employee", "department"]
        c_names = db["column_names_original"] # e.g. [[-1,"*"], [0,"id"], [0,"name"], ...]
        c_types = db["column_types"]          # e.g. ["text","number","text",...]
        pk_ids  = set(db.get("primary_keys", []))
        fk_pairs= db.get("foreign_keys", [])  # e.g. [[col_id, ref_col_id], ...]

        # Build col_id → (table_name, col_name) lookup
        col_lookup = {}
        for cid, (tid, cname) in enumerate(c_names):
            if tid == -1:
                continue
            col_lookup[cid] = (t_names[tid], cname)

        # Build FK map: col_id → "ref_table.ref_col"
        fk_map = {}
        for (src_cid, dst_cid) in fk_pairs:
            if dst_cid in col_lookup:
                ref_table, ref_col = col_lookup[dst_cid]
                fk_map[src_cid] = f"{ref_table}.{ref_col}"

        # Organise per table
        columns = {t: [] for t in t_names}
        pks     = {t: set() for t in t_names}
        fks     = {t: {} for t in t_names}

        for cid, (tid, cname) in enumerate(c_names):
            if tid == -1:
                continue
            tname = t_names[tid]
            ctype = c_types[cid]
            columns[tname].append((cname, ctype.upper()))
            if cid in pk_ids:
                pks[tname].add(cname)
            if cid in fk_map:
                fks[tname][cname] = fk_map[cid]

        schemas[db_id] = dict(tables=t_names, columns=columns, pks=pks, fks=fks)

    return schemas



In [12]:

# ──────────────────────────────────────────────────────────────────────────────
# ENRICHED SCHEMA REPRESENTATIONS  [Improvement 3]
# ──────────────────────────────────────────────────────────────────────────────

def build_table_text(table_name: str, columns: list, pks: set, fks: dict) -> str:
    """
    Builds an enriched text representation of a table for BGE embedding.

    v1.0: "employee: id, name, salary, dept_id"
    v2.0: "employee: id (PK NUMBER), name (TEXT), salary (NUMBER),
                      dept_id (FK→department.id NUMBER)"
    """
    parts = []
    for col_name, col_type in columns:
        tags = []
        if col_name in pks:
            tags.append("PK")
        if col_name in fks:
            tags.append(f"FK→{fks[col_name]}")
        tags.append(col_type)
        parts.append(f"{col_name} ({' '.join(tags)})")
    return f"{table_name}: {', '.join(parts)}"


def build_column_texts(schema: dict) -> tuple[list[str], list[str]]:
    """
    Returns two parallel lists:
        column_texts  — enriched string per column, used for BGE embedding
        column_labels — "table.col" label for each entry
    """
    texts  = []
    labels = []
    for tname in schema["tables"]:
        pks = schema["pks"][tname]
        fks = schema["fks"][tname]
        for col_name, col_type in schema["columns"][tname]:
            tags = []
            if col_name in pks:
                tags.append("PK")
            if col_name in fks:
                tags.append(f"FK→{fks[col_name]}")
            tags.append(col_type)
            texts.append(f"{tname}.{col_name} ({' '.join(tags)})")
            labels.append(f"{tname}.{col_name}")
    return texts, labels



In [14]:

# ──────────────────────────────────────────────────────────────────────────────
# BGE RETRIEVER  [Improvements 1, 2, 3]
# ──────────────────────────────────────────────────────────────────────────────

class SchemaRetriever:
    def __init__(self, model_name: str = CFG.bge_model_name):
        print(f"Loading BGE model: {model_name}")
        self.model = SentenceTransformer(model_name)
        self.model.eval()

    def retrieve(
        self,
        question: str,
        schema: dict,
        threshold: float = CFG.similarity_threshold,
        max_tables: int  = CFG.max_tables_fallback,
        top_k_cols: int  = CFG.top_k_columns,
    ) -> str:
        """
        Returns a CREATE TABLE string for the most relevant columns.

        Pipeline:
          1. Embed question + all column strings with BGE
          2. Score columns via cosine similarity
          3. Take top_k_cols columns; derive their parent tables
          4. For each selected table, filter to relevant columns only
          5. Also apply dynamic table-level threshold as a safety guard  [Impr 1]
          6. Return enriched CREATE TABLE prompt string
        """
        # ── Column-level embedding & scoring  [Improvement 2] ──────────────
        col_texts, col_labels = build_column_texts(schema)
        if not col_texts:
            return ""

        q_emb   = self.model.encode([question], normalize_embeddings=True)
        col_emb = self.model.encode(col_texts,  normalize_embeddings=True)
        col_scores = cosine_similarity(q_emb, col_emb)[0]

        # Top-K columns
        k = min(top_k_cols, len(col_texts))
        top_col_idx = col_scores.argsort()[-k:][::-1]

        # Derive required tables from selected columns
        needed_tables = {col_labels[i].split(".")[0] for i in top_col_idx}

        # ── Dynamic table-level threshold guard  [Improvement 1] ──────────
        # Also embed table-level texts as a secondary filter
        table_texts  = [
            build_table_text(t, schema["columns"][t], schema["pks"][t], schema["fks"][t])
            for t in schema["tables"]
        ]
        tbl_emb    = self.model.encode(table_texts, normalize_embeddings=True)
        tbl_scores = cosine_similarity(q_emb, tbl_emb)[0]

        dynamic_tables = {
            schema["tables"][i]
            for i, s in enumerate(tbl_scores)
            if s >= threshold
        }
        if not dynamic_tables:
            # fallback: best scoring table
            dynamic_tables = {schema["tables"][tbl_scores.argmax()]}

        # Limit to max_tables
        dynamic_tables = set(list(dynamic_tables)[:max_tables])

        # Union: take tables from column-selection AND threshold guard
        selected_tables = (needed_tables | dynamic_tables)
        selected_tables = set(list(selected_tables)[:max_tables])

        # ── Build CREATE TABLE string  [Improvement 3 — enriched] ─────────
        create_stmts = []
        for tname in schema["tables"]:
            if tname not in selected_tables:
                continue
            pks = schema["pks"][tname]
            fks = schema["fks"][tname]
            cols = schema["columns"][tname]

            col_defs = []
            for col_name, col_type in cols:
                constraint = ""
                if col_name in pks:
                    constraint += " PRIMARY KEY"
                if col_name in fks:
                    ref = fks[col_name]
                    constraint += f" REFERENCES {ref}"
                col_defs.append(f"  {col_name} {col_type}{constraint}")

            stmt = f"CREATE TABLE {tname} (\n" + ",\n".join(col_defs) + "\n);"
            create_stmts.append(stmt)

        return "\n".join(create_stmts)



In [15]:

# ──────────────────────────────────────────────────────────────────────────────
# DATASET
# ──────────────────────────────────────────────────────────────────────────────

def build_prompt(question: str, schema_str: str) -> str:
    return f"translate English to SQL: {question} schema: {schema_str}"


class SpiderDataset(Dataset):
    def __init__(
        self,
        data_path: str,
        schemas:   dict,
        retriever: SchemaRetriever,
        tokenizer,
        max_input:  int = CFG.max_input_length,
        max_output: int = CFG.max_output_length,
    ):
        with open(data_path) as f:
            raw = json.load(f)

        self.samples   = []
        self.tokenizer = tokenizer
        self.max_input  = max_input
        self.max_output = max_output

        print(f"Preprocessing {len(raw)} samples from {data_path} …")
        for i, item in enumerate(raw):
            if i % 500 == 0:
                print(f"  {i}/{len(raw)}")
            db_id    = item["db_id"]
            question = item["question"]
            sql      = item["query"]

            if db_id not in schemas:
                continue

            schema_str = retriever.retrieve(question, schemas[db_id])
            prompt     = build_prompt(question, schema_str)
            self.samples.append((prompt, sql))

        print(f"  Done — {len(self.samples)} samples ready.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        prompt, sql = self.samples[idx]

        enc = self.tokenizer(
            prompt,
            max_length=self.max_input,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        
        dec = self.tokenizer(
            sql,
            max_length=self.max_output,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        labels = dec["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100  # ignore padding in loss

        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels":         labels,
        }



In [17]:

# ──────────────────────────────────────────────────────────────────────────────
# TRAINING
# ──────────────────────────────────────────────────────────────────────────────


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load schemas
schemas   = load_schemas(CFG.tables_path)
retriever = SchemaRetriever(CFG.bge_model_name)
retriever.model.to(device)

# Load tokenizer + model
print(f"Loading model: {CFG.model_name}")
tokenizer = T5Tokenizer.from_pretrained(CFG.model_name)
model     = T5ForConditionalGeneration.from_pretrained(CFG.model_name)
model.to(device)


Device: cuda
Loading BGE model: BAAI/bge-base-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading model: google/flan-t5-base


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [ ]:
# from torch.utils.data import Dataset
# import torch

# class DummyDataset(Dataset):
#     def __len__(self):
#         return 8

#     def __getitem__(self, idx):
#         return {
#             "input_ids": torch.tensor([1, 2, 3, 4]),
#             "attention_mask": torch.tensor([1, 1, 1, 1]),
#             "labels": torch.tensor([1, 2, 3, 4]),
#         }

In [ ]:
print(retriever.model.device)

In [ ]:
# Build datasets
train_ds = SpiderDataset(CFG.train_path, schemas, retriever, tokenizer)
dev_ds   = SpiderDataset(CFG.dev_path,   schemas, retriever, tokenizer)
# train_ds = DummyDataset()
# dev_ds   = DummyDataset()

# Training arguments
args = Seq2SeqTrainingArguments(
    output_dir                  = CFG.output_dir,
    num_train_epochs            = CFG.epochs,
    per_device_train_batch_size = CFG.batch_size,
    per_device_eval_batch_size  = CFG.batch_size,
    learning_rate               = CFG.learning_rate,
    warmup_steps                = CFG.warmup_steps,
    weight_decay                = CFG.weight_decay,
    fp16                        = CFG.fp16 and (device == "cuda"),
    predict_with_generate       = True,
    generation_num_beams        = CFG.num_beams,
    generation_max_length       = CFG.max_output_length,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    logging_steps               = 100,
    report_to                   = "none",
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model             = model,
    args              = args,
    train_dataset     = train_ds,
    eval_dataset      = dev_ds,
    processing_class  = tokenizer,
    data_collator     = collator,
    callbacks         = [EarlyStoppingCallback(early_stopping_patience=3)],
)



Preprocessing 7000 samples from /kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/train_spider.json …
  0/7000
  500/7000
  1000/7000
  1500/7000
  2000/7000
  2500/7000
  3000/7000
  3500/7000
  4000/7000
  4500/7000
  5000/7000
  5500/7000
  6000/7000
  6500/7000
  Done — 7000 samples ready.
Preprocessing 1034 samples from /kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/dev.json …
  0/1034
  500/1034
  1000/1034
  Done — 1034 samples ready.
Training …


/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss


In [ ]:
print("Training …")
trainer.train()



In [ ]:
print(f"Saving model to {CFG.output_dir}")
model.push_to_hub("yashwantk05/flan-t5-spider-text2sql")
tokenizer.push_to_hub("yashwantk05/flan-t5-spider-text2sql")

